In [1]:
# ============================================================
# Cell 1: 环境准备
# ============================================================
import os, sys, json
from IPython.display import display, Markdown
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
driver = get_driver()

def h1(t): display(Markdown(f"<h1 style='color:#0f172a;border-bottom:3px solid #0071e3;padding-bottom:0.3rem;'>{t}</h1>"))
def h2(t): display(Markdown(f"<h2 style='color:#1e293b;margin-top:1.5rem;'>▸ {t}</h2>"))
def h3(t): display(Markdown(f"<h3 style='color:#334155;margin-top:1rem;'>• {t}</h3>"))
def hr():    display(Markdown("---"))
def note(t): display(Markdown(f"<blockquote style='border-left:4px solid #94a3b8;color:#64748b;padding-left:0.8rem;'>{t}</blockquote>"))
print("✅ 环境就绪")

✅ 环境就绪


In [2]:
# ============================================================
# Cell 2: 创建专利族 + 临床试验，并挂到项目上
# ============================================================
from src.ci.competitor_profile import create_patent_family
from src.ci.program_service import create_clinical_trial

with driver.session() as s:
    org = s.run("""
        MATCH (o:Organization)
        WHERE EXISTS { MATCH (o)-[:DEVELOPS]->(:DevelopmentProgram) }
        RETURN o.organization_id AS id, o.canonical_name AS name
        ORDER BY o.canonical_name LIMIT 1
    """).single()
    if not org:
        raise RuntimeError("数据库里没有带项目的组织，请先跑 05 notebook")
    print(f"使用组织: {org['name']} ({org['id']})")

with driver.session() as s:
    progs = [r["pid"] for r in s.run("""
        MATCH (ec:EngineeredPhageConstruct)-[:ASSOCIATED_WITH]->(dp:DevelopmentProgram)
        RETURN DISTINCT dp.program_id AS pid
    """)]
    if not progs:
        progs = [r["pid"] for r in s.run("MATCH (d:DevelopmentProgram) RETURN d.program_id AS pid LIMIT 1")]
    print(f"目标项目：{progs}")

patent_ids = []
patent_specs = [
    dict(title="Engineered bacteriophage with expanded host range", priority_date="2022-03-15",
         publication_numbers=["WO2023/123456A1"], jurisdictions=["WO"], legal_status="granted",
         claim_scope_summary="权利要求 1 覆盖尾纤维改造以扩展对 KL47 型 K. pneumoniae 的宿主范围。"),
    dict(title="Phage-encoded biofilm degrading enzyme", priority_date="2023-06-01",
         publication_numbers=["WO2024/654321A1"], jurisdictions=["WO"], legal_status="pending",
         claim_scope_summary="权利要求 1 覆盖在噬菌体基因组中插入生物膜降解酶编码序列。"),
    dict(title="Anti-CRISPR payload for phage therapy", priority_date="2023-11-10",
         publication_numbers=["US2025/0011223A1"], jurisdictions=["US"], legal_status="pending",
         claim_scope_summary="权利要求 1 覆盖装载 anti-CRISPR 蛋白的工程化噬菌体。"),
]
for spec in patent_specs:
    pfid = create_patent_family(driver, **spec, organization_id=org["id"], program_ids=progs)
    patent_ids.append(pfid)
    print(f"✅ 专利族: {pfid}")

trial_ids = []
trial_specs = [
    dict(title="PhageBank-CF Phase 1/2", nct_id="NCT05012345", phase="phase_1_2", status="completed",
         indication="Cystic fibrosis lung infection", sponsor="Adaptive Phage Therapeutics",
         start_date="2021-06-01", completion_date="2023-12-31", enrollment=36),
    dict(title="BX211 Phase 2 in diabetic foot osteomyelitis", nct_id="NCT06098765", phase="phase_2", status="active",
         indication="Diabetic foot osteomyelitis", sponsor="BiomX",
         start_date="2024-02-01", completion_date="2026-06-30", enrollment=120),
]
for spec in trial_specs:
    tid = create_clinical_trial(driver, **spec, organization_id=org["id"], program_id=progs[0])
    trial_ids.append(tid)
    print(f"✅ 临床试验: {tid}")

使用组织: Adaptive Phage Therapeutics (CI:ORG:D1DFF1CF)
目标项目：['CI:PROG:BA36FA6C']
✅ 专利族: CI:PATENT:4E1064E7
✅ 专利族: CI:PATENT:3215DF1E
✅ 专利族: CI:PATENT:A49A8553
✅ 临床试验: CI:TRIAL:EDA2BB68
✅ 临床试验: CI:TRIAL:DE104004


In [3]:
# ============================================================
# Cell 3: 修复孤立节点（构建体/专利/试验 挂到项目）
# ============================================================
with driver.session() as s:
    target = s.run("""
        MATCH (dp:DevelopmentProgram)-[:COVERED_BY]->(:PatentFamily)
        RETURN dp.program_id AS pid, dp.canonical_name AS name LIMIT 1
    """).single()
    target_pid = target["pid"] if target else progs[0]
    print(f"目标项目: {target_pid}")

    orphans = s.run("""
        MATCH (ec:EngineeredPhageConstruct)
        WHERE NOT EXISTS { MATCH (ec)-[:ASSOCIATED_WITH]->(:DevelopmentProgram) }
        RETURN ec.construct_id AS cid
    """).data()
    for o in orphans:
        s.run("MATCH (ec:EngineeredPhageConstruct {construct_id: $cid}) MATCH (dp:DevelopmentProgram {program_id: $pid}) MERGE (ec)-[:ASSOCIATED_WITH]->(dp)", cid=o["cid"], pid=target_pid)
    print(f"✅ 补链孤立构建体: {len(orphans)}")

    patents = s.run("""
        MATCH (pf:PatentFamily)
        WHERE NOT EXISTS { MATCH (:DevelopmentProgram)-[:COVERED_BY]->(pf) }
        RETURN pf.patent_family_id AS pfid
    """).data()
    for p in patents:
        s.run("MATCH (pf:PatentFamily {patent_family_id: $pfid}) MATCH (dp:DevelopmentProgram {program_id: $pid}) MERGE (dp)-[:COVERED_BY]->(pf)", pfid=p["pfid"], pid=target_pid)
    print(f"✅ 补链专利族: {len(patents)}")

    trials = s.run("""
        MATCH (ct:ClinicalTrial)
        WHERE NOT EXISTS { MATCH (:DevelopmentProgram)-[:HAS_TRIAL]->(ct) }
        RETURN ct.trial_id AS tid
    """).data()
    for t in trials:
        s.run("MATCH (ct:ClinicalTrial {trial_id: $tid}) MATCH (dp:DevelopmentProgram {program_id: $pid}) MERGE (dp)-[:HAS_TRIAL]->(ct)", tid=t["tid"], pid=target_pid)
    print(f"✅ 补链临床试验: {len(trials)}")

目标项目: CI:PROG:BA36FA6C
✅ 补链孤立构建体: 9
✅ 补链专利族: 0
✅ 补链临床试验: 0


In [4]:
# ============================================================
# Cell 4: 生成 Competitor Brief，检查 patent_landscape / clinical_trials
# ============================================================
from src.ci.competitor_brief import generate_competitor_brief

brief = generate_competitor_brief(driver, org["id"], days_back=730, persist=True)
print(f"brief_id: {brief.get('brief_id')}")
pl = brief.get("patent_landscape", {})
print(f"专利族总数: {pl.get('total_families')}, 按状态: {pl.get('by_legal_status')}")
ct = brief.get("clinical_trials", {})
print(f"临床试验总数: {ct.get('total_trials')}, 按阶段: {ct.get('by_phase')}")
for s_ in brief.get("recommended_next_steps", []):
    print(f"  - {s_}")

brief_id: CI:BRIEF:A7E9453B
专利族总数: 3, 按状态: {'pending': 2, 'granted': 1}
临床试验总数: 2, 按阶段: {'phase_2': 1, 'phase_1_2': 1}
  - 建议跟进近期重大事件，评估对内部战略的影响
  - 对手拥有 3 个专利族，建议启动 FTO 初筛
  - 对手有 1 项活跃临床试验，建议持续跟踪
  - 识别到 6 个 empty white-space 组合，建议评估进入策略
  - 识别到 9 个 minimal 覆盖组合，建议持续观察
  - 补充数据缺口，完善竞争对手档案


In [5]:
# ============================================================
# Cell 5: IntelligenceQuestion + WatchlistItem
# ============================================================
from src.decision_support.intelligence_question import (
    create_intelligence_question, list_intelligence_questions, link_brief_to_question,
)
from src.decision_support.decision_record import create_watchlist_item, list_watchlist_items

iq1 = create_intelligence_question(driver, question_text="哪些竞争对手正在布局工程化噬菌体治疗 CRKP？",
    question_type="market", requester_id="VP_Strategy", priority="high")
iq2 = create_intelligence_question(driver, question_text="尾纤维工程在扩大宿主范围方面有哪些公开证据？",
    question_type="technology", requester_id="R&D_Lead", priority="medium")
print(f"✅ 问题: {iq1}, {iq2}")

with driver.session() as s:
    strategies = s.run("MATCH (es:EngineeringStrategy) RETURN es.strategy_id AS sid, es.strategy_type AS stype LIMIT 2").data()

wi1 = create_watchlist_item(driver, target_type="organization", target_id=org["id"],
    reason="持有 3 个工程化噬菌体专利族，且有 2 项活跃临床试验", priority="high",
    owner_id="VP_Strategy", review_date="2027-01-01", derived_from_brief_id=brief["brief_id"])
wi2 = create_watchlist_item(driver, target_type="engineering_strategy", target_id=strategies[0]["sid"],
    reason="生物膜破坏策略公开证据增长，需评估内部能力差距", priority="medium",
    owner_id="R&D_Lead", review_date="2027-03-01")
wi3 = create_watchlist_item(driver, target_type="patent_family", target_id=patent_ids[0],
    reason="核心专利族 granted，需启动 FTO 初筛", priority="high",
    owner_id="IP_Lead", review_date="2027-02-01")
print(f"✅ 关注项: {wi1}, {wi2}, {wi3}")

link_brief_to_question(driver, brief["brief_id"], iq1)
print(f"\n✅ 简报 {brief['brief_id']} 已关联到问题 {iq1}")

print("\n=== 情报问题列表 ===")
for q in list_intelligence_questions(driver):
    print(f"  [{q['status']}] {q['question_id']} [{q['question_type']}] {q['question_text'][:40]} 简报数={q['brief_count']}")

✅ 问题: CI:IQ:CE0BAB33, CI:IQ:75D830D4
✅ 关注项: CI:WATCH:3E4E502F, CI:WATCH:1C8D1033, CI:WATCH:C336CC74

✅ 简报 CI:BRIEF:A7E9453B 已关联到问题 CI:IQ:CE0BAB33

=== 情报问题列表 ===
  [open] CI:IQ:75D830D4 [technology] 尾纤维工程在扩大宿主范围方面有哪些公开证据？ 简报数=0
  [open] CI:IQ:CE0BAB33 [market] 哪些竞争对手正在布局工程化噬菌体治疗 CRKP？ 简报数=1


In [6]:
# ============================================================
# Cell 6: 演示三 —— 从情报到决策 (Decision Brief + Review + Record + Use)
# ============================================================
from src.ci.competitor_brief import create_intelligence_product
from src.shared.review import create_review
from src.decision_support.decision_record import create_decision_record
from src.ci.event_service import record_intelligence_use

h1("演示三：情报到决策")

q_text = "是否应优先评估针对目标病原菌的 biofilm_disruption 工程策略？"
iq_id = create_intelligence_question(driver, question_text=q_text, question_type="technology",
    requester_id="VP_Strategy", priority="high")
print(f"① 情报问题: {iq_id}")

with driver.session() as s:
    ec_cnt = s.run("MATCH (es:EngineeringStrategy {strategy_type: 'biofilm_disruption'})<-[:IMPLEMENTS]-(ec) RETURN count(ec) AS c").single()["c"]
    patent_cnt = s.run("MATCH (pf:PatentFamily) RETURN count(pf) AS c").single()["c"]
    trial_cnt = s.run("MATCH (ct:ClinicalTrial) RETURN count(ct) AS c").single()["c"]

exec_summary = (f"本简报围绕问题「{q_text}」组织信息。当前数据库中 {ec_cnt} 个公开构建体覆盖 biofilm_disruption 策略，"
    f"专利族 {patent_cnt} 族，临床试验 {trial_cnt} 项。建议进入 FTO 初筛与可行性评估。")

decision_brief_id = create_intelligence_product(driver, brief_type="technology",
    title=f"决策简报 - {q_text[:40]}", executive_summary=exec_summary,
    as_of_date="2026-09-27", citations=[], competitive_assessment={},
    data_gaps=["部分策略尚缺乏体内数据"],
    recommended_next_steps=["启动 FTO 初筛", "评估内部 biofilm 工程能力差距"])
link_brief_to_question(driver, decision_brief_id, iq_id)
print(f"② 决策简报: {decision_brief_id}")

review_id = create_review(driver, review_type="intelligence_product_review",
    target_object_type="IntelligenceProduct", target_object_id=decision_brief_id,
    reviewer_id="expert_wang", decision="approved", comment="技术论证充分，同意进入 FTO 初筛",
    update_target_status=True)
print(f"③ 专家审核: {review_id}")

dec_id = create_decision_record(driver, brief_id=decision_brief_id, decision_type="evaluate",
    decision_summary="启动 biofilm_disruption 工程策略的可行性评估，并同步启动 FTO 初筛",
    rationale=f"基于 {decision_brief_id} 与专家审核 {review_id}",
    decision_owner="VP_Strategy", review_date="2027-03-01")
print(f"④ 决策记录: {dec_id}")

use_id = record_intelligence_use(driver, product_id=decision_brief_id, consumer_type="Strategy",
    consumer_id="strategy_team", use_purpose="go_no_go_decision",
    context_note="战略团队基于简报做出可行性评估决策", referenced_decision_id=dec_id)
print(f"⑤ 情报消费: {use_id}")

<h1 style='color:#0f172a;border-bottom:3px solid #0071e3;padding-bottom:0.3rem;'>演示三：情报到决策</h1>

① 情报问题: CI:IQ:890C1CEF
② 决策简报: CI:BRIEF:16A1E582
③ 专家审核: REV-06FB36E6
④ 决策记录: CI:DEC:6408E863
⑤ 情报消费: CI:USE:5A48AC29


In [7]:
# ============================================================
# Cell 7: PRD 决策对象缺口（DecisionOption）
# ============================================================
from src.decision_support.decision_record import create_decision_option, list_options_for_brief

opt1 = create_decision_option(driver, brief_id=decision_brief_id, option_type="monitor",
    option_summary="持续监控该对手的临床与专利进展", pros=["成本低", "保持信息更新"],
    cons=["错过窗口期风险"], risk_level="low")
opt2 = create_decision_option(driver, brief_id=decision_brief_id, option_type="evaluate",
    option_summary="启动内部可行性评估（含 FTO 初筛）", pros=["抢占先机", "技术路线验证"],
    cons=["资源投入", "结果不确定"], risk_level="medium")
print(f"✅ DecisionOptions: {opt1}, {opt2}")

print("\n=== Brief 的所有 DecisionOption ===")
for o in list_options_for_brief(driver, decision_brief_id):
    print(f"  {o['option_id']} [{o['option_type']}/{o['risk_level']}] {o['option_summary']}")

✅ DecisionOptions: CI:OPT:1298ADE0, CI:OPT:A69E728D

=== Brief 的所有 DecisionOption ===
  CI:OPT:1298ADE0 [monitor/low] 持续监控该对手的临床与专利进展
  CI:OPT:A69E728D [evaluate/medium] 启动内部可行性评估（含 FTO 初筛）


In [8]:
# ============================================================
# Cell 8: 归因链验证
# ============================================================
with driver.session() as s:
    chain = s.run("""
        MATCH (b:IntelligenceProduct {brief_id: $bid})
        OPTIONAL MATCH (b)-[:ANSWERS]->(iq:IntelligenceQuestion)
        OPTIONAL MATCH (b)<-[:REVIEWS]-(r:Review)
        OPTIONAL MATCH (dr:DecisionRecord)-[:BASED_ON]->(b)
        OPTIONAL MATCH (u:IntelligenceUseEvent)-[:CONSUMES]->(b)
        OPTIONAL MATCH (wi:WatchlistItem)-[:DERIVED_FROM]->(b)
        RETURN iq.question_text AS q, b.brief_id AS brief, r.decision AS review,
               dr.decision_id AS dec, dr.decision_type AS dtype,
               u.consumer_type AS consumer, u.use_purpose AS purpose,
               wi.watchlist_item_id AS watch
    """, bid=decision_brief_id).single()

if chain:
    print("=" * 60)
    print("归因链验证")
    print("=" * 60)
    print(f"问题: {chain['q']}")
    print(f"简报: {chain['brief']} [审核:{chain['review']}]")
    print(f"决策: {chain['dec']} ({chain['dtype']})")
    print(f"消费: {chain['consumer']} / {chain['purpose']}")
    print(f"关注项: {chain['watch']}")
    print("\n✅ Learning Engine 闭环验证通过")

归因链验证
问题: 是否应优先评估针对目标病原菌的 biofilm_disruption 工程策略？
简报: CI:BRIEF:16A1E582 [审核:approved]
决策: CI:DEC:6408E863 (evaluate)
消费: Strategy / go_no_go_decision
关注项: None

✅ Learning Engine 闭环验证通过
